In [73]:
# Set environment variables for proxy
import os

os.environ['http_proxy'] = 'http://127.0.0.1:7890'
os.environ['https_proxy'] = 'http://127.0.0.1:7890'
os.environ['all_proxy'] = 'socks5://127.0.0.1:7890'

In [74]:
#Set environment variables for my notebook
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = r"C:\Users\HP\Desktop\BestTop项目\PTA项目\IMDB Analysis\august-clover-452613-h1-3236960c7686.json"

In [75]:
from google.cloud import bigquery

service_account_path = r"C:\Users\HP\Desktop\BestTop项目\PTA项目\IMDB Analysis\august-clover-452613-h1-3236960c7686.json"

# Create a BigQuery client
client = bigquery.Client.from_service_account_json(service_account_path)

Use magic commands to directly run SQL query in Jupyter Notebook.

In [77]:
%load_ext bigquery_magics

The bigquery_magics extension is already loaded. To reload it, use:
  %reload_ext bigquery_magics


#### Movie Performance Analytics

##### 1.The top 10 highest-rated movies with more than 10,000 votes

In [80]:
%%bigquery df
SELECT r.tconst, primary_title, average_rating
FROM bigquery-public-data.imdb.title_ratings r
JOIN bigquery-public-data.imdb.title_basics b
ON r.tconst = b.tconst
WHERE num_votes >=10000 AND title_type = 'movie'
ORDER BY average_rating DESC
LIMIT 10

Query is running:   0%|          |

Downloading:   0%|          |

In [81]:
df

,tconst,primary_title,average_rating
0,tt33175825,Attack on Titan the Movie: The Last Attack,9.3
1,tt0111161,The Shawshank Redemption,9.3
2,tt0252487,The Chaos Class,9.2
3,tt0068646,The Godfather,9.2
4,tt0259534,Ramayana: The Legend of Prince Rama,9.1
5,tt0167260,The Lord of the Rings: The Return of the King,9.0
6,tt0468569,The Dark Knight,9.0
7,tt16747572,The Silence of Swastika,9.0
8,tt0071562,The Godfather Part II,9.0
9,tt0108052,Schindler's List,9.0


In [82]:
# Save table to Bigquery
#import pandas_gbq
#pandas_gbq.to_gbq(df, "IMDB.top 10 movies",  project_id="august-clover-452613-h1",  if_exists="replace")

##### 2.Over the past decade, the highest average ratings genres

In [84]:
%%bigquery df
SELECT genre, ROUND(AVG(average_rating),2) AS average_rating
FROM bigquery-public-data.imdb.title_ratings r
JOIN bigquery-public-data.imdb.title_basics b
ON r.tconst = b.tconst, -- comma is equal to CROSS JOIN
UNNEST(SPLIT(genres, ',')) AS genre
WHERE start_year>2015 -- this year is 2025
GROUP BY genre
ORDER BY average_rating DESC
LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

In [85]:
df

,genre,average_rating
0,History,7.50
1,Biography,7.39
2,Fantasy,7.35
3,Animation,7.35
4,Game-Show,7.34


#### Talent & Cast Insights

##### 1. Appearance number of actors in the top-rated movies

we assume rate number >8.0 is top-rated

In [89]:
%%bigquery df
SELECT p.nconst,primary_name,COUNT(DISTINCT p.tconst) AS appearance
FROM bigquery-public-data.imdb.title_ratings r
JOIN bigquery-public-data.imdb.title_basics b ON r.tconst = b.tconst
JOIN bigquery-public-data.imdb.title_principals p ON p.tconst = b.tconst
JOIN bigquery-public-data.imdb.name_basics n ON n.nconst=p.nconst
WHERE average_rating >=8.0 AND category IN ('actor', 'actress') AND title_type = 'movie'
GROUP BY p.nconst,primary_name
ORDER BY appearance DESC

Query is running:   0%|          |

Downloading:   0%|          |

In [90]:
df.head()

,nconst,primary_name,appearance
0,nm0004660,Rajkumar,190
1,nm0595934,Mohan Babu,115
2,nm3183374,T.N. Balakrishna,110
3,nm1154608,K.S. Ashwath,107
4,nm0621245,T.R. Narasimharaju,63


##### 2. Directors who consistently produce highly-rated content

we assume rate number >8.0 is highly-rated content.

In [93]:
%%bigquery df
SELECT p.nconst,primary_name, COUNT(DISTINCT p.tconst) AS num_of_works
FROM bigquery-public-data.imdb.title_ratings r
JOIN bigquery-public-data.imdb.title_basics b ON r.tconst = b.tconst
JOIN bigquery-public-data.imdb.title_principals p ON p.tconst = b.tconst
JOIN bigquery-public-data.imdb.name_basics n ON n.nconst=p.nconst
WHERE category ='director'
GROUP BY p.nconst, primary_name
HAVING MIN(average_rating) >= 8.0
ORDER BY num_of_works DESC

Query is running:   0%|          |

Downloading:   0%|          |

In [94]:
df.head()

,nconst,primary_name,num_of_works
0,nm1475126,Cat Santarosa,495
1,nm7786669,Shashank Bharadwaj,376
2,nm11624298,Paulo Viníccius Santos Ferraz,317
3,nm8331592,Christopher Michale Dailey,154
4,nm5092513,Bülent Dogan,121


##### 3. Most prolific actors in each specific genre

we find top 5 prolific actors for each genre.

In [97]:
%%bigquery df
WITH genre_split AS (
  -- split genres strings
  SELECT b.tconst, p.nconst, n.primary_name, genre
  FROM bigquery-public-data.imdb.title_basics b
  JOIN bigquery-public-data.imdb.title_principals p ON b.tconst = p.tconst
  JOIN bigquery-public-data.imdb.name_basics n ON n.nconst = p.nconst
  CROSS JOIN UNNEST(SPLIT(b.genres, ',')) AS genre 
  WHERE p.category IN ('actor', 'actress')),
actor_counts AS (
  -- calculate movie number of each actor in each genre
  SELECT genre, nconst, primary_name, COUNT(DISTINCT tconst) AS content_count,
         RANK() OVER (PARTITION BY genre ORDER BY COUNT(DISTINCT tconst) DESC) AS rank
  FROM genre_split
  GROUP BY genre, nconst, primary_name)
-- select top 5 actors
SELECT genre, nconst, primary_name, content_count
FROM actor_counts
WHERE rank <= 5
ORDER BY genre, rank

Query is running:   0%|          |

Downloading:   0%|          |

In [98]:
df

,genre,nconst,primary_name,content_count
0,Action,nm1296472,Vic Sotto,9972
1,Action,nm0815658,Tito Sotto,9954
2,Action,nm0849028,Mayumi Tanaka,3241
3,Action,nm0847439,Minami Takayama,2521
4,Action,nm0782841,Toshihiko Seki,2514
...,...,...,...,...
136,Western,nm0000790,James Arness,681
137,Western,nm0832065,Milburn Stone,626
138,Western,nm0086469,Amanda Blake,574
139,Western,nm0001446,Michael Landon,456


#### Genre Trends and Popularity

##### 1. In the last 50 years, number of released sci-fi contents in each year

In [101]:
%%bigquery df
SELECT start_year, COUNT(*) AS sci_fi_count
FROM bigquery-public-data.imdb.title_basics
WHERE 'Sci-Fi' IN UNNEST(SPLIT(genres, ','))
AND start_year > 1975
GROUP BY start_year
ORDER BY start_year

Query is running:   0%|          |

Downloading:   0%|          |

In [102]:
df

,start_year,sci_fi_count
0,1976,502
1,1977,417
2,1978,432
3,1979,516
4,1980,419
5,1981,508
6,1982,412
7,1983,447
8,1984,569
9,1985,484


##### 2. Genres that have consistently high average rating(>6.0) in each year

In [104]:
%%bigquery df
WITH genre_year_avg AS
  (SELECT AVG(average_rating) AS average_rating,genre,start_year AS year
  FROM bigquery-public-data.imdb.title_ratings r
  JOIN bigquery-public-data.imdb.title_basics b
  ON r.tconst = b.tconst,
  UNNEST(SPLIT(genres, ',')) AS genre
  GROUP BY genre,start_year),
valid_genres AS
  (SELECT genre
  FROM genre_year_avg
  GROUP BY genre
  HAVING MIN(average_rating)>=6.0)
SELECT g.genre, year, ROUND(average_rating,2) AS avg_rating
FROM genre_year_avg g
JOIN valid_genres v ON g.genre = v.genre
ORDER BY g.genre, year

Query is running:   0%|          |

Downloading:   0%|          |

In [105]:
df

,genre,year,avg_rating
0,Film-Noir,1927,7.50
1,Film-Noir,1928,7.05
2,Film-Noir,1929,6.50
3,Film-Noir,1931,6.37
4,Film-Noir,1932,6.90
5,Film-Noir,1933,6.70
6,Film-Noir,1934,6.30
7,Film-Noir,1935,6.87
8,Film-Noir,1936,6.20
9,Film-Noir,1937,6.61


##### 3. The most common runtime for movies in each genre

In [107]:
%%bigquery df
WITH ranked AS
  (SELECT genre,runtime_minutes, COUNT(*),
    RANK() OVER (PARTITION BY genre ORDER BY COUNT(*) DESC) AS rank
  FROM bigquery-public-data.imdb.title_basics b,
    UNNEST(SPLIT(genres, ',')) AS genre
  WHERE runtime_minutes IS NOT NULL AND title_type = 'movie'
  GROUP BY genre, runtime_minutes
  ORDER BY genre, COUNT(*) DESC)
SELECT genre, runtime_minutes AS most_common_runtime
FROM ranked WHERE rank=1

Query is running:   0%|          |

Downloading:   0%|          |

In [108]:
df

,genre,most_common_runtime
0,Action,90
1,Adult,80
2,Adventure,90
3,Animation,90
4,Biography,90
5,Comedy,90
6,Crime,90
7,Documentary,52
8,Drama,90
9,Family,90


#### TV Series Analysis

##### 1. Which TV shows have the highest-rated episodes?

Since we find 10 rated episodes are too many(>5000), we calculated average episodes rating and choose top 10 long TV series(episodes>=20 and more than 1 season).

In [112]:
%%bigquery df
SELECT 
  bb.primary_title AS show_title,
  COUNT(ba.tconst) AS episode_count,
  ROUND(AVG(r.average_rating),2) AS episode_avg_rating,
  SUM(num_votes) AS total_votes
FROM bigquery-public-data.imdb.title_ratings r
JOIN bigquery-public-data.imdb.title_basics ba ON r.tconst = ba.tconst
JOIN bigquery-public-data.imdb.title_episode e ON e.tconst = ba.tconst
JOIN bigquery-public-data.imdb.title_basics bb ON e.parent_tconst = bb.tconst
WHERE ba.title_type = 'tvEpisode' AND bb.title_type = 'tvSeries'
GROUP BY bb.tconst,bb.primary_title
HAVING COUNT(e.season_number) >= 2 AND COUNT(ba.tconst)>=20 -- only select long tv shows
ORDER BY episode_avg_rating DESC,total_votes DESC
LIMIT 10

Query is running:   0%|          |

Downloading:   0%|          |

In [113]:
df

,show_title,episode_count,episode_avg_rating,total_votes
0,Beautiful Homes & Great Estates,185,10.00,5661
1,MyDestination.TV,131,9.99,4649
2,Recipe TV Featuring the World's Greatest Chefs,701,9.98,24148
3,Kkavyanjali,100,9.98,2393
4,Beautiful Homes,39,9.98,1407
5,Rüzgarli Tepe,160,9.97,1670
6,The White Olive Tree,21,9.97,584
7,Kaise Mujhe Tum Mil Gaye,28,9.97,205
8,LeagueOne: In the Spotlight!,116,9.96,3293
9,Exposed,32,9.94,646


##### 2. What is the average lifespan (number of seasons) of top-rated TV shows?

we assume top-rated is >8.0.

In [116]:
%%bigquery df
WITH seasons_count AS
  (SELECT parent_tconst, COUNT(DISTINCT season_number) AS season_count
  FROM bigquery-public-data.imdb.title_ratings r
  JOIN bigquery-public-data.imdb.title_episode e ON e.parent_tconst = r.tconst
  JOIN bigquery-public-data.imdb.title_basics b ON r.tconst = b.tconst
  WHERE title_type ='tvSeries' AND average_rating >=8.0
  GROUP BY parent_tconst)
SELECT ROUND(AVG(season_count),2) AS avg_season_toptv FROM seasons_count

Query is running:   0%|          |

Downloading:   0%|          |

In [117]:
df

,avg_season_toptv
0,2.29


##### 3. Which genres dominate TV series production?

draw tree map for genres weights

In [120]:
%%bigquery df
WITH genre_count AS (
  SELECT genre, COUNT(DISTINCT tconst) AS genre_series_count
  FROM bigquery-public-data.imdb.title_basics, 
  UNNEST(SPLIT(genres, ',')) AS genre
  WHERE title_type = 'tvSeries'
  GROUP BY genre
),
total_series AS (
  SELECT COUNT(DISTINCT tconst) AS total_series_count
  FROM bigquery-public-data.imdb.title_basics
  WHERE title_type = 'tvSeries'
)
SELECT g.genre, 
  (g.genre_series_count / t.total_series_count) * 100 AS percentage
FROM genre_count g
CROSS JOIN total_series t
ORDER BY percentage DESC

Query is running:   0%|          |

Downloading:   0%|          |

In [121]:
df.head()

,genre,percentage
0,Comedy,22.517460
1,Drama,18.826191
2,Documentary,12.342412
3,Reality-TV,9.254026
4,Talk-Show,8.582560


#### Content Longevity and Relevance

##### 1. Which movies from the 1990s are still popular today?

we find top 10 rated movies in 1990s with their number of votes > 100,000.

In [125]:
%%bigquery df
SELECT primary_title, start_year, average_rating, num_votes
FROM bigquery-public-data.imdb.title_basics b
JOIN bigquery-public-data.imdb.title_ratings r ON b.tconst = r.tconst
WHERE 
    b.start_year BETWEEN 1990 AND 1999
    AND r.num_votes > 100000
    AND title_type = 'movie'
ORDER BY r.average_rating DESC, r.num_votes DESC
LIMIT 10

Query is running:   0%|          |

Downloading:   0%|          |

In [126]:
df

,primary_title,start_year,average_rating,num_votes
0,The Shawshank Redemption,1994,9.3,3018055
1,Schindler's List,1993,9.0,1511037
2,Pulp Fiction,1994,8.9,2315447
3,Fight Club,1999,8.8,2440547
4,Forrest Gump,1994,8.8,2359481
5,The Matrix,1999,8.7,2138668
6,Goodfellas,1990,8.7,1313764
7,Se7en,1995,8.6,1892215
8,The Silence of the Lambs,1991,8.6,1620106
9,Saving Private Ryan,1998,8.6,1557526


##### 2. Are older movies generally rated higher than recent ones?

we find average rating of movies for each year.

In [129]:
%%bigquery df
SELECT start_year,ROUND(AVG(r.average_rating),2) AS avg_rating
FROM bigquery-public-data.imdb.title_basics b
JOIN bigquery-public-data.imdb.title_ratings r ON b.tconst = r.tconst
WHERE title_type = 'movie'
AND start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year

Query is running:   0%|          |

Downloading:   0%|          |

In [130]:
df.head()

,start_year,avg_rating
0,1894,5.40
1,1896,4.00
2,1897,4.80
3,1898,3.58
4,1899,3.80


##### 3.How do ratings change for movies as they age?

In [132]:
%%bigquery df
SELECT 2025 - start_year AS movie_age,
  ROUND(AVG(r.average_rating),2) AS avg_rating
FROM bigquery-public-data.imdb.title_basics b
JOIN bigquery-public-data.imdb.title_ratings r 
ON b.tconst = r.tconst
WHERE title_type = 'movie'
AND start_year IS NOT NULL
GROUP BY movie_age
ORDER BY movie_age

Query is running:   0%|          |

Downloading:   0%|          |

In [133]:
df.head()

,movie_age,avg_rating
0,0,7.11
1,1,6.59
2,2,6.39
3,3,6.35
4,4,6.23


#### Regional or Cultural Insights

##### 1. What are the most popular non-English movies by rating?

In [136]:
%%bigquery df
SELECT title_id, primary_title, language, average_rating
FROM bigquery-public-data.imdb.title_akas a
JOIN bigquery-public-data.imdb.title_basics b ON b.tconst = a.title_id
JOIN bigquery-public-data.imdb.title_ratings r ON b.tconst = r.tconst
WHERE language != 'en' AND title_type = 'movie'
ORDER BY average_rating DESC
LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

In [137]:
df

,title_id,primary_title,language,average_rating
0,tt14946614,(re)começo,es,10.0
1,tt35631332,Given 3: To the Sea,ja,10.0
2,tt15141270,Akshi,hi,9.9
3,tt28552918,Obsolete,ja,9.9
4,tt31113776,Onde as Ondas Quebram,ja,9.9


##### 2. Which countries produce the most highly-rated movies in a specific genre?

For orginal title, we don't have the corresponding region data,so we can only answer that which countries have the most highly-rated movies in a specific genre.

In [140]:
%%bigquery df
WITH genre_region_ranked AS (
SELECT genre,region,ROUND(AVG(average_rating), 2) AS average_rating,
    RANK() OVER (PARTITION BY genre ORDER BY AVG(average_rating) DESC) AS rank
  FROM bigquery-public-data.imdb.title_ratings r
  JOIN bigquery-public-data.imdb.title_basics b ON r.tconst = b.tconst
  JOIN bigquery-public-data.imdb.title_akas a ON b.tconst = a.title_id
  CROSS JOIN UNNEST(SPLIT(genres, ',')) AS genre
  WHERE region IS NOT NULL AND title_type = 'movie'
  GROUP BY genre, region
)
SELECT genre, region AS best_region, average_rating AS avg_rating_in_genre
FROM genre_region_ranked
WHERE rank = 1
ORDER BY genre

Query is running:   0%|          |

Downloading:   0%|          |

In [141]:
df

,genre,best_region,avg_rating_in_genre
0,Action,MV,9.20
1,Adult,SK,8.90
2,Adventure,TZ,9.40
3,Animation,JO,8.90
4,Biography,NI,9.60
5,Comedy,TZ,8.65
6,Crime,QA,9.20
7,Documentary,AN,9.30
8,Drama,BI,9.00
9,Family,MO,9.40


#### Crew Collaboration Analysis

##### 1. Which actor-director pairings produce the highest-rated movies?

In [144]:
%%bigquery df
WITH actor_director_pairing AS (
  SELECT 
    a.tconst AS movie_id,
    na.primary_name AS actor,
    nb.primary_name AS director
  FROM bigquery-public-data.imdb.title_principals a
  JOIN bigquery-public-data.imdb.title_principals b ON a.tconst = b.tconst
  JOIN bigquery-public-data.imdb.name_basics na ON na.nconst = a.nconst 
  JOIN bigquery-public-data.imdb.name_basics nb ON nb.nconst = b.nconst
  WHERE a.category IN ('actor', 'actress') AND b.category = 'director'
),
pair_ratings AS (
  SELECT 
    actor,
    director,
    COUNT(*) AS movie_count,
    SUM(num_votes) AS total_votes,
    AVG(average_rating) AS avg_rating,
    (10000 * 6.8 + SUM(num_votes * average_rating)) / (10000 + SUM(num_votes)) AS weighted_score 
    -- Bayesian Average of movie rating, 6.8 is average rating of IMDB
  FROM bigquery-public-data.imdb.title_ratings r
  JOIN bigquery-public-data.imdb.title_basics b ON r.tconst = b.tconst
  JOIN actor_director_pairing ON movie_id = b.tconst
  WHERE title_type = 'movie'
  GROUP BY actor, director
  HAVING movie_count >= 4 -- at least 4 times collaboration
  AND total_votes >= 10000
  AND actor != director
)
SELECT actor, director, ROUND(avg_rating, 2) AS average_rating, movie_count, total_votes, ROUND(weighted_score, 2) AS weighted_score
FROM pair_ratings
ORDER BY weighted_score DESC
LIMIT 10

Query is running:   0%|          |

Downloading:   0%|          |

In [145]:
df

,actor,director,average_rating,movie_count,total_votes,weighted_score
0,Robert Duvall,Francis Ford Coppola,8.35,4,4265970,8.99
1,Jack Warden,Sidney Lumet,6.82,5,971595,8.89
2,Edward Binns,Sidney Lumet,7.55,4,990271,8.89
3,Sala Baker,Peter Jackson,8.88,4,8132480,8.87
4,Cillian Murphy,Christopher Nolan,8.57,4,8154380,8.70
5,Suzanne Pleshette,Hayao Miyazaki,8.60,4,3597256,8.60
6,Michael Caine,Christopher Nolan,8.53,4,8030958,8.60
7,Kemal Sunal,Ertem Egilmez,8.31,12,198461,8.58
8,Ian McKellen,Peter Jackson,8.28,6,8232498,8.57
9,Sitki Akçatepe,Ertem Egilmez,8.28,5,100061,8.56


##### 2. Who are the most frequent collaborators in a specific genre?

we use 'Sci-Fi' genre as example.

In [148]:
%%bigquery df
WITH genre_movies AS (
  -- Filter specific genre movie
  SELECT tconst
  FROM bigquery-public-data.imdb.title_basics
  WHERE 'Sci-Fi' IN UNNEST(SPLIT(genres, ',')) 
  AND title_type = 'movie'
),
collaborations AS (
  -- Get the actor & director of movies
  SELECT a.tconst, na.primary_name AS actor, nb.primary_name AS director
  FROM bigquery-public-data.imdb.title_principals a
  JOIN bigquery-public-data.imdb.title_principals b ON a.tconst = b.tconst
  JOIN bigquery-public-data.imdb.name_basics na ON na.nconst = a.nconst
  JOIN bigquery-public-data.imdb.name_basics nb ON nb.nconst = b.nconst
  JOIN genre_movies gm ON a.tconst = gm.tconst
  WHERE a.category IN ('actor', 'actress') AND b.category = 'director'
)
-- Calculate collaboration number
SELECT actor, director, COUNT(*) AS collaboration_count
FROM collaborations
WHERE actor!=director
GROUP BY actor, director
ORDER BY collaboration_count DESC
LIMIT 10

Query is running:   0%|          |

Downloading:   0%|          |

In [149]:
df

,actor,director,collaboration_count
0,Jeff Kirkendall,Mark Polonia,13
1,Michael G. Kaiser,Christopher R. Mihm,13
2,Kenji Sahara,Ishirô Honda,12
3,Yoshio Tsuchiya,Ishirô Honda,11
4,Akihiko Hirata,Ishirô Honda,11
5,Edson Camacho,BC Fourteen,10
6,Marco Guzmán,BC Fourteen,10
7,Yoshifumi Tajima,Ishirô Honda,9
8,Robert Downey Jr.,Joe Russo,9
9,Akira Takarada,Ishirô Honda,9


#### Sequels and Franchise Analytics

##### 1. Do sequels generally perform better or worse than the original movie?

In [152]:
%%bigquery df
WITH possible_sequels AS (
  SELECT 
    b.tconst, primary_title, start_year, average_rating,
    -- find movies that is possibly sequels
    CASE 
      WHEN primary_title LIKE '% _' OR primary_title LIKE '% II' 
        OR primary_title LIKE '% III' OR primary_title LIKE '% IV' OR primary_title LIKE '% V'
        OR primary_title LIKE '% VI' OR primary_title LIKE '% VII' OR primary_title LIKE '% X'
        OR primary_title LIKE '% _: %' OR primary_title LIKE '% __: %'
        OR primary_title LIKE '% Part %' AND primary_title NOT LIKE '% Part 1%' 
           AND primary_title NOT LIKE '% Part I%' AND primary_title NOT LIKE '% Part One%'
      THEN 'sequel'
      ELSE 'original'
    END AS movie_type
  FROM bigquery-public-data.imdb.title_basics b
  JOIN bigquery-public-data.imdb.title_ratings r ON b.tconst = r.tconst
  WHERE title_type = 'movie'
)
-- calculate average rating of sequels and original movies
SELECT 
  movie_type, 
  COUNT(*) AS movie_count, 
  ROUND(AVG(average_rating), 2) AS avg_rating
FROM possible_sequels
GROUP BY movie_type

Query is running:   0%|          |

Downloading:   0%|          |

In [153]:
df

,movie_type,movie_count,avg_rating
0,original,320294,6.16
1,sequel,6548,5.89


##### 2. Which franchises have the highest average ratings across their movies?

In [155]:
%%bigquery df
WITH movie_franchises AS (
  SELECT 
    b.tconst, primary_title, average_rating,
    -- We select frachises for some classic movies
    CASE 
      WHEN primary_title LIKE 'Harry Potter%' THEN 'Harry Potter'
      WHEN primary_title LIKE 'The Lord of the Rings%' OR primary_title LIKE 'The Hobbit%' THEN 'Lord of the Rings'
      WHEN primary_title LIKE 'Star Wars%' THEN 'Star Wars'
      WHEN primary_title LIKE 'Spider-Man%' THEN 'Spider-Man'
      WHEN primary_title LIKE 'Avengers%' OR primary_title LIKE 'Iron Man%' OR primary_title LIKE 'Captain America%' THEN 'Marvel'
      WHEN primary_title LIKE 'Mission: Impossible%' THEN 'Mission: Impossible'
    END AS franchise
  FROM bigquery-public-data.imdb.title_basics b
  JOIN bigquery-public-data.imdb.title_ratings r ON b.tconst = r.tconst
  WHERE title_type = 'movie'
)
-- calculate average rating of each selected franchise
SELECT franchise, COUNT(*) AS movie_count, ROUND(AVG(average_rating), 2) AS avg_rating
FROM movie_franchises
WHERE franchise IS NOT NULL
GROUP BY franchise
ORDER BY avg_rating DESC

Query is running:   0%|          |

Downloading:   0%|          |

In [156]:
df

,franchise,movie_count,avg_rating
0,Harry Potter,14,7.69
1,Lord of the Rings,11,7.49
2,Mission: Impossible,7,7.19
3,Spider-Man,26,7.09
4,Star Wars,25,7.00
5,Marvel,20,6.49


#### Career Trajectory of Artists

##### 1. How does an actor’s average movie rating evolve throughout their career?

In [159]:
%%bigquery df
WITH actor_movies AS (
  -- calculate average movies rating of each actor for each year
    SELECT 
        p.nconst AS actor_id,
        n.primary_name AS actor_name,
        b.start_year,
        ROUND(AVG(r.average_rating), 2) AS avg_rating_per_year,
        COUNT(*) AS movies_in_year
    FROM bigquery-public-data.imdb.title_principals p
    JOIN bigquery-public-data.imdb.title_basics b ON p.tconst = b.tconst
    JOIN bigquery-public-data.imdb.title_ratings r ON b.tconst = r.tconst
    JOIN bigquery-public-data.imdb.name_basics n ON p.nconst = n.nconst
    WHERE p.category IN ('actor', 'actress')
    AND b.title_type = 'movie'
    GROUP BY p.nconst, n.primary_name, b.start_year
)
SELECT 
-- calculate cumulative average movie rating of each actor over the years
    actor_name,
    start_year,
    movies_in_year,
    avg_rating_per_year,
    ROUND(AVG(avg_rating_per_year) OVER (PARTITION BY actor_name ORDER BY start_year ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) 
          AS cumulative_avg_rating
FROM actor_movies
ORDER BY actor_name, start_year

Query is running:   0%|          |

Downloading:   0%|          |

In [160]:
df.head()

,actor_name,start_year,movies_in_year,avg_rating_per_year,cumulative_avg_rating
0,$2 Tony,2010,1,5.5,5.5
1,'Amanda Rose' Amanda Rose Mattingly,2023,1,4.0,4.0
2,'Ana Ika,2023,1,5.3,5.3
3,'Angry' Joe Cleary,2003,1,9.0,9.0
4,'Baby' Carmen De Rue,1914,2,5.9,5.9


##### 2. Which directors improved their ratings over the years?

In [162]:
%%bigquery df
WITH director_movies AS (
  -- calculate average movies rating of each director for each year
    SELECT 
        p.nconst AS director_id,
        n.primary_name AS director_name,
        b.start_year,
        ROUND(AVG(r.average_rating), 2) AS avg_rating_per_year,
        COUNT(*) AS movies_in_year
    FROM bigquery-public-data.imdb.title_principals p
    JOIN bigquery-public-data.imdb.title_basics b ON p.tconst = b.tconst
    JOIN bigquery-public-data.imdb.title_ratings r ON b.tconst = r.tconst
    JOIN bigquery-public-data.imdb.name_basics n ON p.nconst = n.nconst
    WHERE p.category = 'director'
    AND b.title_type = 'movie'
    GROUP BY p.nconst, n.primary_name, b.start_year
),
rating_trend AS (
  -- calculate cumulative average movie rating of each director over the years
    SELECT 
        director_name,
        start_year,
        avg_rating_per_year,
        ROUND(AVG(avg_rating_per_year) OVER (PARTITION BY director_name ORDER BY start_year ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) 
            AS cumulative_avg_rating
    FROM director_movies
)
SELECT 
    director_name,
    MIN(cumulative_avg_rating) AS first_cum_avg_rating,
    MAX(cumulative_avg_rating) AS last_cum_avg_rating,
    ROUND(MAX(cumulative_avg_rating) - MIN(cumulative_avg_rating), 2) AS rating_improvement
FROM rating_trend
GROUP BY director_name
HAVING MAX(cumulative_avg_rating) > MIN(cumulative_avg_rating)  -- filter rating improved over the years
ORDER BY rating_improvement DESC
LIMIT 10

Query is running:   0%|          |

Downloading:   0%|          |

In [163]:
df

,director_name,first_cum_avg_rating,last_cum_avg_rating,rating_improvement
0,Jeff Kaufman,1.50,6.57,5.07
1,Lothar Mendes,1.00,5.83,4.83
2,Neal 'Buboy' Tan,2.00,6.80,4.80
3,Jack Foster,3.82,8.60,4.78
4,Luiz de Barros,2.00,6.58,4.58
5,A.V. Bramble,1.50,6.00,4.50
6,Chris Atkins,1.90,6.38,4.48
7,Maria Ramos,1.60,6.02,4.42
8,Jastis Arimba,1.20,5.52,4.32
9,Pete Guzzo,4.70,9.00,4.30
